# Brain Tumor Segmentation - Basic U-Net (PyTorch, BraTS 2021)


## Setup (Miniconda + RTX 5070)
```bat
conda activate brats
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
pip install nibabel opencv-python matplotlib tqdm torchinfo scipy pandas notebook ipykernel
```
Open in VS Code, pick the **brats** kernel.

### Imports & configuration
**Output:** versions + run settings.

In [ ]:
import os, glob, random, json, csv
import numpy as np
import pandas as pd
import nibabel as nib
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
try:
    from torchinfo import summary as torchinfo_summary
    HAS_TORCHINFO = True
except Exception:
    HAS_TORCHINFO = False

# ---- paths ----
DATA_ROOTS = [r"D:\archive", "/kaggle/input"]
PROCESSED_DIR = r"D:\brats_processed"
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./brats_out"

# ---- data ----
NUM_PATIENTS = 500            # memory-safe: try up to 1251. More = better, more disk + time.
VAL_FRAC, TEST_FRAC = 0.20, 0.20
N_VAL_PATIENTS = round(NUM_PATIENTS * VAL_FRAC)
N_TEST_PATIENTS = round(NUM_PATIENTS * TEST_FRAC)
IMG_SIZE = 128                # original used 256; 128 keeps it light. Set 256 to match exactly.
NEIGHBOR_SLICES = 2
KEEP_EMPTY_FRACTION = 0.10
MAX_SLICES_PER_PATIENT = 100
MIN_BRAIN_FRACTION = 0.02
STORE_FLOAT16 = True
FORCE_PREPROCESS = False

# ---- model / training (same recipe as the original notebook) ----
BASE_FILTERS = 64
IN_CHANNELS = 4              # FLAIR, T1, T1ce, T2
OUT_CHANNELS = 3            # WT, TC, ET
EPOCHS = 200
PATIENCE = 15               # early stopping
BATCH_SIZE = 16             # original 64; lower for the 12 GB RTX 5070. Raise if memory allows.
LR = 1e-4
WEIGHT_DECAY = 1e-4
RLROP_FACTOR = 0.5          # ReduceLROnPlateau
RLROP_PATIENCE = 4
MIN_LR = 1e-7
NUM_WORKERS = 0             # keep 0 on Windows notebooks; Kaggle/Linux: try 2-4
SEED = 42

MODALITIES = ["flair", "t1", "t1ce", "t2"]
REGIONS = ["WT", "TC", "ET"]
DISK_DTYPE = np.float16 if STORE_FLOAT16 else np.float32

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

print("torch", torch.__version__)
print(f"Patients {NUM_PATIENTS}: {NUM_PATIENTS-N_VAL_PATIENTS-N_TEST_PATIENTS} train / {N_VAL_PATIENTS} val / {N_TEST_PATIENTS} test")
print(f"img={IMG_SIZE} | batch={BATCH_SIZE} | base_filters={BASE_FILTERS} | epochs={EPOCHS} (patience {PATIENCE}) | LR={LR}")
print("Model=U-Net++ (SE decoder) | Loss=Dice only | Metrics=Dice/IoU/Precision/Recall")
print("\n========== [OK] Step 1 complete ==========")

### GPU check
**Output:** GPU name + real compute test.

In [ ]:
def get_device():
    if not torch.cuda.is_available():
        print("[warn] CUDA not available -> CPU (slow)."); return torch.device("cpu")
    print("GPU:", torch.cuda.get_device_name(0), "(sm_%d%d)" % torch.cuda.get_device_capability(0))
    try:
        _ = torch.randn(1024, 1024, device="cuda") @ torch.randn(1024, 1024, device="cuda")
        torch.cuda.synchronize(); print("GPU compute test: OK")
    except RuntimeError as e:
        raise SystemExit("GPU compute failed - install the cu128 build.\n" + str(e))
    return torch.device("cuda")

device = get_device()
print("\n========== [OK] Step 2 complete ==========")

## 1. Load Dataset
Find BraTS patient folders under `D:\archive`.
**Output:** number of patients.

In [ ]:
def find_patient_dirs(roots):
    for root in roots:
        if not os.path.isdir(root):
            continue
        flairs = glob.glob(os.path.join(root, "**", "*_flair.nii.gz"), recursive=True)
        dirs = sorted({os.path.dirname(f) for f in flairs})
        if dirs:
            print(f"Found {len(dirs)} patients under {root}"); return dirs
    raise SystemExit("No patients found. Unzip the data into D:\\archive or edit DATA_ROOTS.")

patients = find_patient_dirs(DATA_ROOTS)
print("Example:", os.path.basename(patients[0]))
print("\n========== [OK] Step 3 complete ==========")

## 2. Dataset structure & channels
| Channel | Modality | Shows | Importance |
|--------|----------|-------|-----------|
| 0 | FLAIR | fluid-attenuated | whole tumor / edema |
| 1 | T1 | anatomy | reference |
| 2 | T1ce | contrast | enhancing tumor / core |
| 3 | T2 | fluid-sensitive | edema / extent |
| - | seg | labels 1/2/4 | ground-truth |
**Output:** shapes, labels, one slice across channels.

In [ ]:
def load_vol(pdir, m):
    pid = os.path.basename(pdir)
    return nib.load(os.path.join(pdir, f"{pid}_{m}.nii.gz")).get_fdata().astype(np.float32)

demo = patients[0]; vols = {m: load_vol(demo, m) for m in MODALITIES}; seg = load_vol(demo, "seg")
print("Patient:", os.path.basename(demo), "| seg labels:", np.unique(seg).astype(int).tolist())
for m in MODALITIES: print(f"  {m:5s} {vols[m].shape}")
z = max(range(seg.shape[2]), key=lambda k: (seg[:, :, k] > 0).sum())
fig, ax = plt.subplots(1, 5, figsize=(15, 3))
for j, m in enumerate(MODALITIES):
    ax[j].imshow(vols[m][:, :, z], cmap="gray"); ax[j].set_title(m); ax[j].axis("off")
ax[4].imshow(seg[:, :, z], cmap="nipy_spectral"); ax[4].set_title("seg"); ax[4].axis("off")
plt.tight_layout(); plt.show()
print("\n========== [OK] Step 4 complete ==========")

## 3. Preprocessing (channel-based) -> disk cache
Each modality z-scored independently on brain voxels; tumor slices + neighbours + a few empties kept, resized, and written to `PROCESSED_DIR`. Runs once; reruns skip automatically.
**Output:** per-channel normalisation check.

In [ ]:
def znorm_channel(vol):
    brain = vol > 0
    if brain.sum() == 0: return vol
    vol = (vol - vol[brain].mean()) / (vol[brain].std() + 1e-8); vol[~brain] = 0.0
    return vol

def rz_img(sl): return cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
def rz_msk(sl): return cv2.resize(sl.astype(np.uint8), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

def select_slices(seg, mods, Z):
    brain_ok = [(mods[0][:, :, z] != 0).mean() >= MIN_BRAIN_FRACTION for z in range(Z)]
    tumor = [z for z in range(Z) if brain_ok[z] and seg[:, :, z].max() > 0]
    ctx = set()
    for tz in tumor:
        for dz in range(-NEIGHBOR_SLICES, NEIGHBOR_SLICES + 1):
            zz = tz + dz
            if 0 <= zz < Z and brain_ok[zz]: ctx.add(zz)
    ctx -= set(tumor)
    empties = [z for z in range(Z) if brain_ok[z] and z not in set(tumor) | ctx]
    random.shuffle(empties); empties = empties[:int(len(empties) * KEEP_EMPTY_FRACTION)]
    keep = list(tumor) + sorted(ctx) + empties
    if MAX_SLICES_PER_PATIENT and len(keep) > MAX_SLICES_PER_PATIENT: keep = keep[:MAX_SLICES_PER_PATIENT]
    return sorted(set(keep))

print("Channel-based normalisation (brain voxels):")
for m in MODALITIES:
    raw = vols[m]; b = raw > 0; nz = znorm_channel(raw.copy()); bn = nz != 0
    print(f"  {m:5s} raw {raw[b].mean():7.1f}/{raw[b].std():6.1f} -> norm {nz[bn].mean():5.2f}/{nz[bn].std():4.2f}")
print("\n========== [OK] Step 5 complete ==========")

### Build the on-disk slice cache (once)
**Output:** slice counts per split + disk used.

In [ ]:
def preprocess_to_disk(patient_dirs):
    img_dir = os.path.join(PROCESSED_DIR, "images"); msk_dir = os.path.join(PROCESSED_DIR, "masks")
    meta_path = os.path.join(PROCESSED_DIR, "meta.json")
    if (not FORCE_PREPROCESS) and os.path.exists(meta_path):
        meta = json.load(open(meta_path))
        if meta.get("num_patients") == NUM_PATIENTS and meta.get("img_size") == IMG_SIZE:
            print("Found existing cache -> skipping preprocessing."); return meta["splits"]
    os.makedirs(img_dir, exist_ok=True); os.makedirs(msk_dir, exist_ok=True)
    pdirs = list(patient_dirs); random.shuffle(pdirs); pdirs = pdirs[:NUM_PATIENTS]
    val_set = set(os.path.basename(p) for p in pdirs[:N_VAL_PATIENTS])
    test_set = set(os.path.basename(p) for p in pdirs[N_VAL_PATIENTS:N_VAL_PATIENTS + N_TEST_PATIENTS])
    splits = {"train": [], "val": [], "test": []}
    for pdir in tqdm(pdirs, desc="Preprocessing to disk"):
        pid = os.path.basename(pdir)
        mods = [znorm_channel(load_vol(pdir, m)) for m in MODALITIES]
        seg = load_vol(pdir, "seg"); Z = seg.shape[2]
        which = "val" if pid in val_set else "test" if pid in test_set else "train"
        for z in select_slices(seg, mods, Z):
            sid = f"{pid}_z{z:03d}"
            img = np.stack([rz_img(m[:, :, z]) for m in mods], 0).astype(DISK_DTYPE)
            s = seg[:, :, z]
            mask = np.stack([rz_msk(np.isin(s, [1, 2, 4])), rz_msk(np.isin(s, [1, 4])), rz_msk(s == 4)], 0).astype(np.uint8)
            np.save(os.path.join(img_dir, sid + ".npy"), img); np.save(os.path.join(msk_dir, sid + ".npy"), mask)
            splits[which].append(sid)
    json.dump({"num_patients": NUM_PATIENTS, "img_size": IMG_SIZE, "splits": splits}, open(meta_path, "w"))
    return splits

splits = preprocess_to_disk(patients)
img_dir = os.path.join(PROCESSED_DIR, "images"); msk_dir = os.path.join(PROCESSED_DIR, "masks")
n = sum(len(v) for v in splits.values())
per = 4 * IMG_SIZE * IMG_SIZE * (2 if STORE_FLOAT16 else 4) + 3 * IMG_SIZE * IMG_SIZE
for k in ["train", "val", "test"]: print(f"{k.upper():5s}: {len(splits[k]):6d} slices")
print(f"Total {n} slices ~ {n*per/1e9:.1f} GB at {PROCESSED_DIR}")
print("\n========== [OK] Step 6 complete ==========")

### Datasets + DataLoaders (synchronized augmentation)
Train-only: flips, 90-degree rotations, scale/zoom, gamma, intensity - applied identically to image and mask.
**Output:** one training batch shape.

In [ ]:
def _fix(a, H, W):
    ch, nh, nw = a.shape; out = np.zeros((ch, H, W), a.dtype)
    y0, x0 = max((nh - H) // 2, 0), max((nw - W) // 2, 0); oy, ox = max((H - nh) // 2, 0), max((W - nw) // 2, 0)
    hh, ww = min(nh, H), min(nw, W); out[:, oy:oy + hh, ox:ox + ww] = a[:, y0:y0 + hh, x0:x0 + ww]; return out

def _scale(x, y, s):
    H, W = x.shape[1:]; nh, nw = max(1, round(H * s)), max(1, round(W * s))
    xr = np.stack([cv2.resize(x[c], (nw, nh), interpolation=cv2.INTER_LINEAR) for c in range(x.shape[0])], 0)
    yr = np.stack([cv2.resize(y[c], (nw, nh), interpolation=cv2.INTER_NEAREST) for c in range(y.shape[0])], 0)
    return _fix(xr, H, W), _fix(yr, H, W)

def _gamma(x, g):
    m = x != 0
    if m.sum() == 0: return x
    v = x[m]; lo, hi = v.min(), v.max()
    if hi > lo: x[m] = (np.clip((v - lo) / (hi - lo), 0, 1) ** g) * (hi - lo) + lo
    return x

def augment(x, y):
    if random.random() < 0.5: x = x[:, :, ::-1]; y = y[:, :, ::-1]
    if random.random() < 0.5: x = x[:, ::-1, :]; y = y[:, ::-1, :]
    k = random.randint(0, 3)
    if k: x = np.rot90(x, k, (1, 2)); y = np.rot90(y, k, (1, 2))
    x, y = np.ascontiguousarray(x), np.ascontiguousarray(y)
    if random.random() < 0.5: x, y = _scale(x, y, random.uniform(0.85, 1.15))
    if random.random() < 0.5: x = _gamma(x, random.uniform(0.7, 1.5))
    x = x * random.uniform(0.9, 1.1) + random.uniform(-0.1, 0.1)
    return x, y

class DiskSliceDS(Dataset):
    def __init__(self, ids, aug=False): self.ids, self.aug = ids, aug
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        sid = self.ids[i]
        x = np.load(os.path.join(img_dir, sid + ".npy")).astype(np.float32)
        y = np.load(os.path.join(msk_dir, sid + ".npy")).astype(np.float32)
        if self.aug: x, y = augment(x, y)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y))

pin = device.type == "cuda"
train_loader = DataLoader(DiskSliceDS(splits["train"], True),  BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=pin)
val_loader   = DataLoader(DiskSliceDS(splits["val"], False),   BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
test_loader  = DataLoader(DiskSliceDS(splits["test"], False),  BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
xb, yb = next(iter(train_loader)); print("Batch images:", tuple(xb.shape), "| masks:", tuple(yb.shape))
print("\n========== [OK] Step 7 complete ==========")

## Basic U-Net (PyTorch)


In [ ]:
class ConvBlock(nn.Module):
    """Two 3x3 convolutions, each followed by BatchNorm and ReLU."""
    def __init__(self, cin, cout):
        super().__init__(); self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    """Basic 2D U-Net: encoder-decoder with plain skip connections (baseline)."""
    def __init__(self, in_ch=4, out_ch=3, base=64):
        super().__init__()
        f = [base, base * 2, base * 4, base * 8, base * 16]
        self.pool = nn.MaxPool2d(2)
        # encoder
        self.e0 = ConvBlock(in_ch, f[0])
        self.e1 = ConvBlock(f[0], f[1])
        self.e2 = ConvBlock(f[1], f[2])
        self.e3 = ConvBlock(f[2], f[3])
        # bottleneck
        self.b  = ConvBlock(f[3], f[4])
        # decoder (transpose-conv up + concat skip + conv block)
        self.u3 = nn.ConvTranspose2d(f[4], f[3], 2, stride=2); self.d3 = ConvBlock(f[4], f[3])
        self.u2 = nn.ConvTranspose2d(f[3], f[2], 2, stride=2); self.d2 = ConvBlock(f[3], f[2])
        self.u1 = nn.ConvTranspose2d(f[2], f[1], 2, stride=2); self.d1 = ConvBlock(f[2], f[1])
        self.u0 = nn.ConvTranspose2d(f[1], f[0], 2, stride=2); self.d0 = ConvBlock(f[1], f[0])
        self.out = nn.Conv2d(f[0], out_ch, 1)
    def _up_cat(self, up, skip):
        if up.shape[-2:] != skip.shape[-2:]:
            up = F.interpolate(up, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return torch.cat([up, skip], 1)
    def forward(self, x):
        e0 = self.e0(x)
        e1 = self.e1(self.pool(e0))
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        b  = self.b(self.pool(e3))
        d3 = self.d3(self._up_cat(self.u3(b),  e3))
        d2 = self.d2(self._up_cat(self.u2(d3), e2))
        d1 = self.d1(self._up_cat(self.u1(d2), e1))
        d0 = self.d0(self._up_cat(self.u0(d1), e0))
        return self.out(d0)   # logits (sigmoid applied in loss/metrics)

model = UNet(IN_CHANNELS, OUT_CHANNELS, BASE_FILTERS).to(device)
print("=============== Basic U-Net (baseline) ===============")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
if HAS_TORCHINFO:
    torchinfo_summary(model, input_size=(1, IN_CHANNELS, IMG_SIZE, IMG_SIZE), col_names=("output_size", "num_params"), depth=2)
print("\n========== [OK] Step 8 complete ==========")


## Loss - Dice (only)
Aggregate soft Dice loss - no BCE term.
**Output:** perfect prediction -> loss ~ 0.

In [ ]:
class DiceLoss(nn.Module):
    def forward(self, logits, y, eps=1e-6):
        p = torch.sigmoid(logits)
        inter = (p * y).sum(); denom = p.sum() + y.sum()
        return 1 - (2 * inter + eps) / (denom + eps)

_y = (torch.rand(2, 3, 32, 32) > 0.5).float()
print("Dice loss, perfect prediction:", round(DiceLoss()((_y - 0.5) * 20, _y).item(), 4), "(expect ~0)")
print("\n========== [OK] Step 9 complete ==========")

## Metrics - Dice, IoU, Precision, Recall
Same set as the original, accumulated over the whole set (micro-averaged), reported per region.
**Output:** perfect prediction -> Dice = IoU = 1.

In [ ]:
@torch.no_grad()
def accumulate(logits, y, acc, thr=0.5):
    p = (torch.sigmoid(logits) > thr).float()
    acc["tp"] += (p * y).sum((0, 2, 3)).cpu()
    acc["fp"] += (p * (1 - y)).sum((0, 2, 3)).cpu()
    acc["fn"] += ((1 - p) * y).sum((0, 2, 3)).cpu()
    return acc

def new_acc(): return {"tp": torch.zeros(3), "fp": torch.zeros(3), "fn": torch.zeros(3)}

def metrics_from(acc, eps=1e-6):
    tp, fp, fn = acc["tp"], acc["fp"], acc["fn"]
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    prec = (tp + eps) / (tp + fp + eps)
    rec = (tp + eps) / (tp + fn + eps)
    return dice, iou, prec, rec

_a = accumulate((_y - 0.5) * 20, _y, new_acc()); _d, _u, _pr, _rc = metrics_from(_a)
print("perfect -> Dice", [round(v, 3) for v in _d.tolist()], "| IoU", [round(v, 3) for v in _u.tolist()])
print("\n========== [OK] Step 10 complete ==========")

## Training
AdamW (lr 1e-4, wd 1e-4), BCE+Dice, mixed precision. Callbacks matched to the original: **best-checkpoint** (val Dice), **early stopping** (patience 15, restore best), **ReduceLROnPlateau** (factor 0.5, patience 4, min_lr 1e-7), **CSV log**, **NaN guard**.
**Output:** per-epoch train/val loss + val Dice/IoU + LR, curves.

In [ ]:
def train_model(model, tr_loader, va_loader, device):
    torch.backends.cudnn.benchmark = True
    loss_fn = DiceLoss()
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=RLROP_FACTOR, patience=RLROP_PATIENCE, min_lr=MIN_LR)
    amp = device.type == "cuda"; scaler = torch.amp.GradScaler("cuda", enabled=amp)
    csv_path = os.path.join(OUT_DIR, "training_log.csv")
    with open(csv_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "train_loss", "val_loss", "val_dice", "val_iou", "lr"])
    hist = {k: [] for k in ["train", "val", "dice", "iou"]}
    best = -1.0; best_epoch = 0; no_improve = 0

    for ep in range(1, EPOCHS + 1):
        model.train(); run = 0.0
        for x, y in tqdm(tr_loader, leave=False, desc=f"epoch {ep}/{EPOCHS} [train]"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=amp):
                loss = loss_fn(model(x), y)
            if not torch.isfinite(loss):                    # NaN guard (TerminateOnNaN)
                print("Non-finite loss - stopping."); return hist
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item() * x.size(0)
        tr_loss = run / len(tr_loader.dataset)

        model.eval(); run = 0.0; acc = new_acc()
        with torch.no_grad():
            for x, y in tqdm(va_loader, leave=False, desc=f"epoch {ep}/{EPOCHS} [val]"):
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                with torch.amp.autocast("cuda", enabled=amp):
                    logits = model(x); loss = loss_fn(logits, y)
                run += loss.item() * x.size(0); acc = accumulate(logits.float(), y, acc)
        va_loss = run / len(va_loader.dataset)
        dice, iou, _, _ = metrics_from(acc); mdice, miou = dice.mean().item(), iou.mean().item()
        sched.step(va_loss)                                  # ReduceLROnPlateau on val loss
        lr_now = opt.param_groups[0]["lr"]
        for k, v in zip(hist, [tr_loss, va_loss, mdice, miou]): hist[k].append(v)
        with open(csv_path, "a", newline="") as f:
            csv.writer(f).writerow([ep, f"{tr_loss:.5f}", f"{va_loss:.5f}", f"{mdice:.5f}", f"{miou:.5f}", f"{lr_now:.2e}"])

        star = ""
        if mdice > best:
            best = mdice; best_epoch = ep; no_improve = 0
            torch.save({"model": model.state_dict(), "base": BASE_FILTERS, "in_ch": IN_CHANNELS, "out_ch": OUT_CHANNELS, "epoch": ep}, os.path.join(OUT_DIR, "best_model.pt"))
            star = "  <- best saved"
        else:
            no_improve += 1; star = f"  (no improve {no_improve}/{PATIENCE})"
        print(f"Epoch {ep:03d}/{EPOCHS} | train {tr_loss:.4f} | val {va_loss:.4f} | val Dice {mdice:.3f} | val IoU {miou:.3f} | lr {lr_now:.1e}{star}")
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {ep}. Restoring best weights (epoch {best_epoch})."); break

    ep_range = range(1, len(hist["train"]) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(ep_range, hist["train"], label="train loss"); plt.plot(ep_range, hist["val"], label="val loss")
    plt.title("Dice loss"); plt.xlabel("epoch"); plt.legend()
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=120); plt.show()
    print(f"\nBest mean val Dice: {best:.3f} (epoch {best_epoch})")
    return hist

import time as _t; _t0=_t.time()

history = train_model(model, train_loader, val_loader, device)

TRAIN_TIME_SEC=_t.time()-_t0; print(f"Training time: {TRAIN_TIME_SEC:.1f} s")
print("\n========== [OK] Step 11 complete ==========")

## Test evaluation - Dice, IoU, Precision, Recall (held-out patients)
**Output:** per-region table.

In [ ]:
ckpt = torch.load(os.path.join(OUT_DIR, "best_model.pt"), map_location=device)
best_model = model.to(device)
best_model.load_state_dict(ckpt["model"]); best_model.eval()
print(f"Loaded best model from epoch {ckpt['epoch']}")

acc = new_acc()
with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Testing"):
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = best_model(x)
        acc = accumulate(logits.float(), y, acc)
dice, iou, prec, rec = metrics_from(acc)
print("\n================= TEST RESULTS =================")
print("Region |  Dice  |  IoU   | Precision | Recall")
print("-------+--------+--------+-----------+-------")
for i, r in enumerate(REGIONS):
    print(f"{r:6s} | {dice[i]:.3f}  | {iou[i]:.3f}  |   {prec[i]:.3f}   | {rec[i]:.3f}")
print(f"{'Mean':6s} | {dice.mean():.3f}  | {iou.mean():.3f}  |   {prec.mean():.3f}   | {rec.mean():.3f}")
print("\n========== [OK] Step 12 complete ==========")

## Visualise predictions
**Output:** FLAIR / ground-truth / prediction for a few test slices.

In [ ]:
def visualize(model, ids, device, n_show=6):
    model.eval(); ids = list(ids); random.shuffle(ids); picks = []
    for sid in ids:
        if np.load(os.path.join(msk_dir, sid + ".npy"))[0].sum() > 50: picks.append(sid)
        if len(picks) >= n_show: break
    fig, axes = plt.subplots(len(picks), 3, figsize=(9, 3 * len(picks)))
    if len(picks) == 1: axes = axes[None, :]
    for r, sid in enumerate(picks):
        x = np.load(os.path.join(img_dir, sid + ".npy")).astype(np.float32)
        y = np.load(os.path.join(msk_dir, sid + ".npy")).astype(np.float32)
        with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            pred = (torch.sigmoid(model(torch.from_numpy(x).unsqueeze(0).to(device)))[0, 0] > 0.5).cpu().numpy()
        gt = y[0]; flair = x[0]; d = (2 * (pred * gt).sum() + 1e-6) / (pred.sum() + gt.sum() + 1e-6)
        axes[r, 0].imshow(flair, cmap="gray"); axes[r, 0].set_title("FLAIR")
        axes[r, 1].imshow(flair, cmap="gray"); axes[r, 1].imshow(gt, alpha=0.4, cmap="Reds"); axes[r, 1].set_title("Ground truth (WT)")
        axes[r, 2].imshow(flair, cmap="gray"); axes[r, 2].imshow(pred, alpha=0.4, cmap="Reds"); axes[r, 2].set_title(f"Prediction (Dice {d:.2f})")
        for a in axes[r]: a.axis("off")
    fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "predictions.png"), dpi=120); plt.show()

visualize(best_model, splits["test"], device)
print("\n========== [OK] Step 13 complete ==========")

## Notes
- **U-Net++ with SE blocks in the decoder**, Dice-only loss, Dice/IoU/Precision/Recall metrics, AdamW, early stopping + ReduceLROnPlateau, on BraTS (4 in / 3 out) at D:\archive.
- **To match the original exactly:** set `IMG_SIZE = 256` and lower `BATCH_SIZE` (8 or 4) so it fits the 12 GB card. Expect much higher disk use and slower epochs.
- **Memory-safe:** slices stream from `PROCESSED_DIR`; only the current batch is in RAM, so you can raise `NUM_PATIENTS` up to 1251.
- **Out of GPU memory?** lower `BATCH_SIZE`. **Slow epochs on Windows?** keep `NUM_WORKERS=0`; on Kaggle/Linux use 2-4.

## Experiment Logging (metrics, params, FLOPs, GPU, HD95 -> Excel/CSV)


In [ ]:
# ============================================================
# Experiment logging: params, FLOPs, inference time, GPU memory,
# convergence epoch, training time, full test metrics + HD95.
# Saves one appended row to experiments.xlsx and experiments.csv.
# ============================================================
import os, csv, time, math, json, shutil, datetime
import numpy as np, torch

MODEL_NAME = "Basic 2D U-Net"
OPTIMIZER  = "Adam"
EXP_TAG    = f"{MODEL_NAME}_{OPTIMIZER}".replace(" ", "")

# ---- ensure GPU is used ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type=="cuda" else "CPU only")

# ---- make sure we have the best model in eval mode ----
try:
    _bm = best_model
except NameError:
    ckpt = torch.load(os.path.join(OUT_DIR, "best_model.pt"), map_location=device)
    model.load_state_dict(ckpt["model"]); _bm = model
_bm = _bm.to(device).eval()

# ---- parameter count ----
n_params = sum(p.numel() for p in _bm.parameters())
n_params_M = n_params / 1e6

# ---- FLOPs (optional: thop or ptflops) ----
flops_G = None
_dummy = torch.randn(1, IN_CHANNELS, IMG_SIZE, IMG_SIZE, device=device)
try:
    from thop import profile
    _f, _ = profile(_bm, inputs=(_dummy,), verbose=False)
    flops_G = _f / 1e9
except Exception:
    try:
        from ptflops import get_model_complexity_info
        _macs, _ = get_model_complexity_info(_bm, (IN_CHANNELS, IMG_SIZE, IMG_SIZE),
                                             as_strings=False, print_per_layer_stat=False, verbose=False)
        flops_G = 2 * _macs / 1e9
    except Exception:
        flops_G = float("nan")   # pip install thop  (or ptflops) to enable

# ---- inference time (ms per image, batch=1) ----
_bm.eval()
with torch.no_grad():
    for _ in range(5):  # warm-up
        _ = _bm(_dummy)
    if device.type == "cuda": torch.cuda.synchronize()
    _t0 = time.time(); _N = 50
    for _ in range(_N):
        _ = _bm(_dummy)
    if device.type == "cuda": torch.cuda.synchronize()
    infer_ms = (time.time() - _t0) / _N * 1000.0

# ---- peak GPU memory (MB) for one inference ----
gpu_mem_MB = float("nan")
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = _bm(_dummy)
    torch.cuda.synchronize()
    gpu_mem_MB = torch.cuda.max_memory_allocated() / 1e6

# ---- HD95 helper (95th-percentile symmetric surface distance, in pixels) ----
def _hd95_2d(pred, gt):
    try:
        from scipy.ndimage import binary_erosion, distance_transform_edt
    except Exception:
        return None
    pred = pred.astype(bool); gt = gt.astype(bool)
    if pred.sum() == 0 or gt.sum() == 0:
        return None
    sp = pred ^ binary_erosion(pred); sg = gt ^ binary_erosion(gt)
    if sp.sum() == 0 or sg.sum() == 0:
        return None
    dt_g = distance_transform_edt(~sg); dt_p = distance_transform_edt(~sp)
    d = np.concatenate([dt_g[sp], dt_p[sg]])
    return float(np.percentile(d, 95))

# ---- full test-set metrics (Dice/IoU/Precision/Recall per region) + HD95 ----
acc = new_acc()
hd95_lists = [[] for _ in range(len(REGIONS))]
_bm.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device); y = y.to(device)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = _bm(x)
        acc = accumulate(logits.float(), y, acc)
        p = (torch.sigmoid(logits.float()) > 0.5).cpu().numpy().astype(np.uint8)
        g = y.cpu().numpy().astype(np.uint8)
        for b in range(p.shape[0]):
            for r in range(len(REGIONS)):
                h = _hd95_2d(p[b, r], g[b, r])
                if h is not None: hd95_lists[r].append(h)
dice, iou, prec, rec = metrics_from(acc)
dice, iou, prec, rec = dice.tolist(), iou.tolist(), prec.tolist(), rec.tolist()
hd95 = [ (float(np.mean(v)) if len(v) else float("nan")) for v in hd95_lists ]

# ---- training-curve summary from training_log.csv ----
conv_epoch = total_epochs = None
best_tr = best_val = best_dice = best_iou = best_lr = float("nan")
csv_path = os.path.join(OUT_DIR, "training_log.csv")
if os.path.exists(csv_path):
    rows = list(csv.DictReader(open(csv_path)))
    total_epochs = len(rows)
    if rows:
        best_row = max(rows, key=lambda r: float(r["val_dice"]))
        conv_epoch = int(best_row["epoch"])
        best_tr  = float(best_row["train_loss"]); best_val = float(best_row["val_loss"])
        best_dice = float(best_row["val_dice"]);  best_iou = float(best_row["val_iou"])
        best_lr  = float(best_row["lr"])
train_time = float(TRAIN_TIME_SEC) if "TRAIN_TIME_SEC" in globals() else float("nan")

# ---- copy the two output pictures under an experiment-specific name ----
for src in ["training_curves.png", "predictions.png"]:
    sp = os.path.join(OUT_DIR, src)
    if os.path.exists(sp):
        shutil.copy(sp, os.path.join(OUT_DIR, f"{EXP_TAG}_{src}"))

# ---- assemble one row ----
row = {
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "model": MODEL_NAME, "optimizer": OPTIMIZER,
    "params_M": round(n_params_M, 3),
    "FLOPs_G": (round(flops_G, 3) if flops_G==flops_G else "NA"),
    "infer_ms": round(infer_ms, 3),
    "gpu_mem_MB": (round(gpu_mem_MB, 1) if gpu_mem_MB==gpu_mem_MB else "NA"),
    "convergence_epoch": conv_epoch, "total_epochs": total_epochs,
    "train_time_s": (round(train_time,1) if train_time==train_time else "NA"),
    "train_loss": round(best_tr,5), "val_loss": round(best_val,5),
    "val_dice": round(best_dice,5), "val_iou": round(best_iou,5), "lr": best_lr,
    "WT_Dice": round(dice[0],4), "TC_Dice": round(dice[1],4), "ET_Dice": round(dice[2],4),
    "WT_IoU": round(iou[0],4),  "TC_IoU": round(iou[1],4),  "ET_IoU": round(iou[2],4),
    "WT_Prec": round(prec[0],4),"TC_Prec": round(prec[1],4),"ET_Prec": round(prec[2],4),
    "WT_Rec": round(rec[0],4),  "TC_Rec": round(rec[1],4),  "ET_Rec": round(rec[2],4),
    "WT_HD95": (round(hd95[0],3) if hd95[0]==hd95[0] else "NA"),
    "TC_HD95": (round(hd95[1],3) if hd95[1]==hd95[1] else "NA"),
    "ET_HD95": (round(hd95[2],3) if hd95[2]==hd95[2] else "NA"),
    "Mean_Dice": round(float(np.mean(dice)),4), "Mean_IoU": round(float(np.mean(iou)),4),
    "Mean_Prec": round(float(np.mean(prec)),4), "Mean_Rec": round(float(np.mean(rec)),4),
}

# ---- append to CSV ----
csv_out = os.path.join(OUT_DIR, "experiments.csv")
write_header = not os.path.exists(csv_out)
with open(csv_out, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(row.keys()))
    if write_header: w.writeheader()
    w.writerow(row)

# ---- append to Excel (openpyxl) ----
try:
    from openpyxl import Workbook, load_workbook
    xlsx_out = os.path.join(OUT_DIR, "experiments.xlsx")
    if os.path.exists(xlsx_out):
        wb = load_workbook(xlsx_out); ws = wb.active
    else:
        wb = Workbook(); ws = wb.active; ws.append(list(row.keys()))
    ws.append(list(row.values())); wb.save(xlsx_out)
    print("Saved to", xlsx_out)
except Exception as e:
    print("openpyxl not available (pip install openpyxl) - CSV saved instead:", e)

print("Saved to", csv_out)
print("\n================= EXPERIMENT SUMMARY =================")
for k, v in row.items(): print(f"{k:18s}: {v}")
print("\n========== [OK] experiment logged ==========")
